In [5]:
import pandas as pd

# FIX: Added 'r' before the string to handle Windows backslashes correctly
df = pd.read_csv(r'C:\Users\user\Desktop\1_darkom-analytics-pipeline\data\raw\darkom_annonces.csv')

# Select only numerical columns for outlier detection
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

outlier_data = []

for col in numerical_cols:
    # Calculate IQR bounds
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Identify outliers
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    count = len(outliers)
    percentage = (count / len(df)) * 100
    
    outlier_data.append({
        'Column': col,
        'Outlier Count': count,
        'Percentage (%)': round(percentage, 2),
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2)
    })

# Convert to DataFrame for a clean, sorted display
outlier_summary = pd.DataFrame(outlier_data).sort_values(by='Outlier Count', ascending=False)
outlier_summary.style.background_gradient(cmap='Reds', subset=['Outlier Count'])

,Column,Outlier Count,Percentage (%),Lower Bound,Upper Bound
0,prix,101,6.700000,-2699636.930000,4530192.640000
4,etage,51,3.380000,-4.500000,7.500000
1,surface,40,2.650000,-127.600000,425.690000
3,nb_salles_bain,34,2.250000,-0.500000,3.500000
5,annee_construction,31,2.060000,1981.500000,2033.500000
2,nb_chambres,16,1.060000,-3.500000,8.500000


In [1]:
import pandas as pd

# Load the dataset (using your raw data path)
df = pd.read_csv(r'C:\Users\user\Desktop\1_darkom-analytics-pipeline\data\raw\darkom_annonces.csv')

# Calculate missing values
missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

# Combine into a clean summary DataFrame
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_percentage
})

# Filter to show only columns that actually have missing values, sorted by highest missing
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

# Display with a nice visual gradient if there are missing values
if not missing_summary.empty:
    print(f"Total rows in dataset: {len(df)}")
    display(missing_summary.style.background_gradient(cmap='Oranges', subset=['Missing Count']))
else:
    print("Perfect! No missing values found in any column.")

Total rows in dataset: 1508


,Missing Count,Percentage (%)
quartier,415,27.519894
etage,232,15.384615
annee_construction,204,13.527851
nb_chambres,129,8.554377
nb_salles_bain,104,6.896552
date_publication,76,5.039788
type_bien,38,2.519894
transaction,38,2.519894


In [2]:
import pandas as pd
import numpy as np

# Load your dataset
path = r'C:\Users\user\Desktop\1_darkom-analytics-pipeline\data\raw\darkom_annonces.csv'
df = pd.read_csv(path)

print(f"=== Starting Data Audit (Total Rows: {len(df)}) ===\n")

# --- 1. MISSING VALUES & EMPTY STRINGS ---
print("1. [Missing Data Audit]")
# Catch both true NaNs and columns that just have empty spaces/strings
missing_summary = {}
for col in df.columns:
    true_nulls = df[col].isnull().sum()
    empty_strings = df[col].astype(str).str.strip().eq('').sum() if df[col].dtype == 'object' else 0
    total_bad = true_nulls + empty_strings
    if total_bad > 0:
        missing_summary[col] = {
            'Missing/Empty Count': total_bad,
            'Percentage (%)': round((total_bad / len(df)) * 100, 2)
        }

if missing_summary:
    display(pd.DataFrame(missing_summary).T.sort_values(by='Missing/Empty Count', ascending=False))
else:
    print("✓ No missing values or empty strings detected.")
print("-" * 50)

# --- 2. DUPLICATE RECORDS ---
print("2. [Duplicate Records Audit]")
exact_duplicates = df.duplicated().sum()
id_duplicates = df.duplicated(subset=['annonce_id']).sum() if 'annonce_id' in df.columns else 0

print(f"• Exact duplicate rows: {exact_duplicates}")
print(f"• Duplicate 'annonce_id' entries: {id_duplicates}")
print("-" * 50)

# --- 3. LOGICAL & BUSINESS RULE CONTRADICTIONS ---
print("3. [Real Estate Business Logic Audit]")
contradictions = []

# Rule A: Apartments or Villas with 0 rooms or 0 bathrooms
zero_rooms = df[(df['type_bien'].isin(['Appartement', 'Villa'])) & ((df['nb_chambres'] == 0) | (df['nb_salles_bain'] == 0))]
contradictions.append({'Issue': 'Properties with 0 rooms/bathrooms', 'Count': len(zero_rooms)})

# Rule B: Missing essential pricing or surface data (zeros or negatives)
invalid_metrics = df[(df['prix'] <= 0) | (df['surface'] <= 0)]
contradictions.append({'Issue': 'Price or Surface is 0 or negative', 'Count': len(invalid_metrics)})

# Rule C: Year of construction is in the future or impossibly old (e.g., 0)
invalid_years = df[(df['annee_construction'] > 2026) | (df['annee_construction'] < 1800)]
contradictions.append({'Issue': 'Invalid/Future construction year', 'Count': len(invalid_years)})

# Rule D: Transaction type conflicts (e.g., Rental priced like a sale or vice versa)
# Based on your business rules: Location if < 50,000 DH, Vente if > 50,000 DH
wrong_rentals = df[(df['transaction'] == 'Location') & (df['prix'] > 50000)]
wrong_sales = df[(df['transaction'] == 'Vente') & (df['prix'] <= 50000)]
contradictions.append({'Issue': 'Rentals with price > 50k DH (Likely Sales)', 'Count': len(wrong_rentals)})
contradictions.append({'Issue': 'Sales with price <= 50k DH (Likely Rentals)', 'Count': len(wrong_sales)})

# Display logical issues
logic_df = pd.DataFrame(contradictions)
display(logic_df[logic_df['Count'] > 0])
print("-" * 50)

# --- 4. DATA TYPE & TEXT CONSISTENCY ---
print("4. [Text Uniformity Audit]")
# Check if structural columns have messy case inconsistencies (e.g., 'RABAT' vs 'Rabat')
text_cols = ['ville', 'type_bien', 'transaction']
for col in text_cols:
    if col in df.columns:
        unique_values = df[col].dropna().unique()
        print(f"• Unique entries in '{col}': {list(unique_values[:10])} (Total unique: {len(unique_values)})")

=== Starting Data Audit (Total Rows: 1508) ===

1. [Missing Data Audit]


,Missing/Empty Count,Percentage (%)
quartier,415.0,27.52
etage,232.0,15.38
annee_construction,204.0,13.53
nb_chambres,129.0,8.55
nb_salles_bain,104.0,6.90
date_publication,76.0,5.04
type_bien,38.0,2.52
transaction,38.0,2.52


--------------------------------------------------
2. [Duplicate Records Audit]
• Exact duplicate rows: 8
• Duplicate 'annonce_id' entries: 8
--------------------------------------------------
3. [Real Estate Business Logic Audit]


,Issue,Count
3,Rentals with price > 50k DH (Likely Sales),4
4,Sales with price <= 50k DH (Likely Rentals),9


--------------------------------------------------
4. [Text Uniformity Audit]
• Unique entries in 'ville': ['Meknès', 'Kenitra', 'Oujda', 'Tanger', 'Rabat', 'Agadir', 'TANGER', 'Fès', 'MEKNÈS', 'oujda'] (Total unique: 31)
• Unique entries in 'type_bien': ['Terrain', 'Appartement', 'Villa', 'Bureau', 'Duplex'] (Total unique: 5)
• Unique entries in 'transaction': ['Vente', 'Location'] (Total unique: 2)


In [1]:
import os
from utils.db_connect import get_db_connection

def analyze_raw_transactions():
    print("=========================================================================")
    # Translates to: Raw Transaction Values Exploratory Analysis
    print("🔍 ANALYSE EXPLORATOIRE DES VALEURS DE TRANSACTION RAW")
    print("=========================================================================\n")
    
    # Connect to your PostgreSQL Instance
    conn = get_db_connection()
    cursor = conn.cursor()
    
    try:
        # Aggregation query to find all unique variants and their frequency
        query = """
            SELECT 
                transaction AS raw_value,
                COUNT(*) AS record_count,
                ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) as percentage
            FROM staging.raw_annonces
            GROUP BY transaction
            ORDER BY record_count DESC;
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        print(f"📊 Found {len(results)} distinct entry types in the raw 'transaction' column:\n")
        print(f"{'RAW VALUE IN CSV':<30} | {'ROW COUNT':<12} | {'PERCENTAGE':<10}")
        print("-" * 60)
        
        for row in results:
            # Handle display formatting if the raw value is an actual database NULL
            display_value = "NULL (Missing/Blank Value)" if row[0] is None else repr(row[0])
            print(f"{display_value:<30} | {row[1]:<12} | {row[2]}%")
            
    except Exception as e:
        print(f"❌ Failed to query staging layer: {e}")
    finally:
        cursor.close()
        conn.close()

if __name__ == "__main__":
    analyze_raw_transactions()

ModuleNotFoundError: No module named 'psycopg2'